---
title: Temporal split
execute:
  enabled: true
---

`q.dataset.TemporalSplit` creates one fixed chronological split. Partitions can be defined either by row counts or proportions, or by inclusive index boundaries. This notebook starts with a proportional train/validation/test split, then demonstrates exact boundaries, a bounded test period, and a custom split name.

## Create the smallest useful dataset

A temporal split needs only an ordered `Dataset`. Targets, sample weights, and row metadata are optional, so this example uses the final 12 sessions of QRT's bundled SPY history and one feature column.

`metadata` is reserved for row-aligned context that is not a model input, such as a label's end time, symbol, or event identifier. `TemporalSplit` does not read it; it partitions rows from the dataset index. Metadata becomes necessary for operations such as `PurgedTimeSeriesSplit`, which uses label end times to remove overlapping training labels.

In [1]:
import qrt as q

In [2]:
spy = q.data.datasets.load("spy").tail(12) # Just load last 12 rows of the SPY dataset for demonstration purposes.
dataset = q.dataset.Dataset(X=spy[["close"]])

assert dataset.index.is_monotonic_increasing # We ensure that the index is sorted in increasing order, which is important for temporal datasets.
assert not dataset.is_split # We ensure that the dataset is not split yet, as we will perform a temporal split next.
dataset.X

,close
datetime,
2026-07-02,744.780029
2026-07-06,751.280029
2026-07-07,747.710022
2026-07-08,745.400024
2026-07-09,751.710022
2026-07-10,754.950012
2026-07-13,749.169983
2026-07-14,751.830017
2026-07-15,754.809998


## 1. Split by proportion or row count

Use size arguments when the desired partition lengths matter more than exact dates. Floats represent proportions of the full dataset; integers represent exact row counts. Partitions are contiguous, preserve chronological order, and consume every row.

| Parameter | Default | Effect |
|---|---:|---|
| `train_size` | `None` | Training proportion or row count |
| `validation_size` | `None` | Validation proportion or row count; `None` omits validation |
| `test_size` | `None` | Test proportion or row count |

Float sizes must sum to `1`, and integer sizes must sum to the dataset length. One of `train_size` or `test_size` may be omitted to receive the remainder. Size arguments cannot be combined with boundary arguments.

Proportional boundaries are rounded to whole rows while preserving the requested cumulative allocation. For this 12-row dataset, 70/15/15 produces 8 train, 2 validation, and 2 test rows.

In [3]:
proportional = dataset.split(
    q.dataset.TemporalSplit(
        train_size=0.70,
        validation_size=0.15,
        test_size=0.15,
    )
)

assert len(proportional.train) == 8
assert len(proportional.validation) == 2
assert len(proportional.test) == 2
assert len(proportional.partition("excluded")) == 0

q.dataset.split_diagnostics(proportional, "default")

,role,rows,proportion,start,end
partition,,,,,
train,fit,8,0.666667,2026-07-02,2026-07-14
validation,evaluate,2,0.166667,2026-07-15,2026-07-16
test,holdout,2,0.166667,2026-07-17,2026-07-20
excluded,excluded,0,0.000000,NaT,NaT


## 2. Split with `train_end`

Use inclusive index boundaries when partition cutoffs must align with specific dates or index values. In boundary mode, `train_end` is required and the row at that index remains in `train`. The defaults provide the simplest train/test holdout:

| Parameter | Default | Effect |
|---|---:|---|
| `train_end` | required | Last row assigned to `train` |
| `validation_end` | `None` | Omits the validation partition |
| `test_end` | `None` | Extends `test` through the final dataset row |
| `name` | `"default"` | Names the split inside the default scheme |

In [4]:
train_end = dataset.index[5]
basic = dataset.split(q.dataset.TemporalSplit(train_end=train_end))

assert not dataset.is_split
assert basic.is_split # We test to see that the dataset is split after performing the temporal split.
assert basic.train.index.max() == train_end
assert basic.test.index.min() > train_end

membership = basic.splits["default"].split().membership
basic.X.assign(partition=membership)


,close,partition
datetime,,
2026-07-02,744.780029,train
2026-07-06,751.280029,train
2026-07-07,747.710022,train
2026-07-08,745.400024,train
2026-07-09,751.710022,train
2026-07-10,754.950012,train
2026-07-13,749.169983,test
2026-07-14,751.830017,test
2026-07-15,754.809998,test


### Inspect partition views

The minimum call creates one fixed train/test assignment. `train_end` belongs to `train`; because `validation_end=None`, the next row begins `test`; because `test_end=None`, test continues through the final row.

```{mermaid}
flowchart LR
    TRAIN["train<br/>rows 1-6<br/>includes train_end"] -->|next row| TEST["test<br/>rows 7-12<br/>through dataset end"]
    TEST -.-> EXCLUDED["excluded<br/>0 rows"]
    classDef train fill:#d8f3dc,stroke:#2d6a4f,color:#16382a
    classDef test fill:#ffefc1,stroke:#9c6b00,color:#4d3500
    classDef excluded fill:#eceff1,stroke:#6b7280,color:#374151
    class TRAIN train
    class TEST test
    class EXCLUDED excluded
```

Splitting returns a new dataset and leaves the original unsplit. Conventional attributes and named indexing return lazy partition views over the split dataset. Diagnostics summarize every partition, including the empty `excluded` partition.

In [5]:
assert basic["train"].index.equals(basic.train.index)
assert basic["test"].index.equals(basic.test.index)

q.dataset.split_diagnostics(basic, "default")

,role,rows,proportion,start,end
partition,,,,,
train,fit,6,0.5,2026-07-02,2026-07-10
test,holdout,6,0.5,2026-07-13,2026-07-20
excluded,excluded,0,0.0,NaT,NaT


## 3. Add `validation_end`

Set `validation_end` to create a three-way boundary split. This boundary is also inclusive: validation begins immediately after `train_end` and ends at `validation_end`; test begins on the next row.

In [6]:
validation_end = dataset.index[7]
three_way = dataset.split(
    q.dataset.TemporalSplit(
        train_end=dataset.index[4],
        validation_end=validation_end,
    )
)

assert three_way.validation.index.max() == validation_end
assert three_way.test.index.min() > validation_end
q.dataset.split_diagnostics(three_way, "default")

,role,rows,proportion,start,end
partition,,,,,
train,fit,5,0.416667,2026-07-02,2026-07-09
validation,evaluate,3,0.250000,2026-07-10,2026-07-14
test,holdout,4,0.333333,2026-07-15,2026-07-20
excluded,excluded,0,0.000000,NaT,NaT


## 4. Bound the test period with `test_end`

By default, test consumes every row after the preceding boundary. Set `test_end` when later observations must remain outside this experiment. Rows after that inclusive boundary are retained in the dataset and assigned to `excluded`.

In [7]:
test_end = dataset.index[9]
bounded = dataset.split(
    q.dataset.TemporalSplit(
        train_end=dataset.index[3],
        validation_end=dataset.index[6],
        test_end=test_end,
    )
)

assert bounded.test.index.max() == test_end
assert bounded.partition("excluded").index.tolist() == dataset.index[10:].tolist()
q.dataset.split_diagnostics(bounded, "default")

,role,rows,proportion,start,end
partition,,,,,
train,fit,4,0.333333,2026-07-02,2026-07-08
validation,evaluate,3,0.250000,2026-07-09,2026-07-13
test,holdout,3,0.250000,2026-07-14,2026-07-16
excluded,excluded,2,0.166667,2026-07-17,2026-07-20


## 5. Give the split a descriptive `name`

A `TemporalSplit` is attached as the dataset's `default` scheme because it represents one fixed holdout design. Changing `name` identifies the split within that scheme, which is useful when recording the purpose or date of a holdout.

In [8]:
named = dataset.split(
    q.dataset.TemporalSplit(
        train_end=dataset.index[7],
        name="final_holdout",
    )
)

assert tuple(named.splits) == ("default",)
assert named.splits["default"].split().name == "final_holdout"
q.dataset.split_diagnostics(named, "default", "final_holdout")

,role,rows,proportion,start,end
partition,,,,,
train,fit,8,0.666667,2026-07-02,2026-07-14
test,holdout,4,0.333333,2026-07-15,2026-07-20
excluded,excluded,0,0.000000,NaT,NaT


## Split rules

Choose `TemporalSplit` when one fixed chronological holdout is sufficient. The dataset index must be monotonic increasing, and size arguments cannot be combined with boundary arguments.

In size mode, use either all proportions or all row counts. Requested partitions must be non-empty and consume the full dataset. In boundary mode, `validation_end` must be later than `train_end`, and `test_end` must be later than the preceding boundary.

Use `TimeSeriesSplit` when evaluation must repeat across multiple walk-forward folds, or `PurgedTimeSeriesSplit` when overlapping label horizons require leakage controls.